# Clinical Concept Extraction Agent Pipeline
This notebook demonstrates a two-agent pipeline that takes a medical conversation transcript and processes it:
1. **Extraction Agent**: Uses Gemini 1.5 Flash to extract and classify medical entities into a highly structured JSON schema based on the SOAP framework.
2. **Final Agent**: Synthesizes the structured JSON and the original transcript into a `CaseSheetSummary` format.
3. **PDF Generation**: Converts the synthesized data into a well-formatted PDF report matching a strict hospital case sheet template using `fpdf2`.


In [ ]:
!pip install -q fpdf2 pydantic langchain_google_vertexai python-dotenv


In [ ]:
from pydantic import BaseModel, Field
from typing import List, Optional
from langchain_google_vertexai import ChatVertexAI
from langchain_core.prompts import ChatPromptTemplate
from dotenv import load_dotenv
import os
from fpdf import FPDF

load_dotenv()


In [ ]:
# --- Agent 1: Structured Data Extraction Schema ---

class EncounterDetails(BaseModel):
    date: Optional[str] = Field(description="Date of the encounter")
    time: Optional[str] = Field(description="Time of the encounter")
    location_type: Optional[str] = Field(description="Location type (e.g., telehealth, clinic, ER)")

class Participants(BaseModel):
    patient_name: Optional[str] = Field(description="Patient name or demographics")
    provider_name: Optional[str] = Field(description="Provider name and role")

class AdministrativeData(BaseModel):
    encounter_details: EncounterDetails
    participants: Participants

class HPI(BaseModel):
    onset: Optional[str] = Field(description="When did the symptom start?")
    provocation_palliation: Optional[str] = Field(description="What makes it better or worse?")
    quality: Optional[str] = Field(description="How does it feel? (e.g., dull, sharp, burning)")
    region_radiation: Optional[str] = Field(description="Where is it located? Does it radiate?")
    severity: Optional[str] = Field(description="How bad is it? (e.g., 6/10)")
    time: Optional[str] = Field(description="How often does it happen or how long does it last?")

class Subjective(BaseModel):
    chief_complaint: str = Field(description="The primary reason for the visit in the patient's own words.")
    hpi: HPI = Field(description="History of Present Illness")
    past_medical_history: List[str] = Field(description="Pre-existing conditions mentioned.")
    medications: List[str] = Field(description="Current prescriptions and over-the-counter drugs.")
    allergies: List[str] = Field(description="Any mentioned allergies or adverse reactions.")
    social_family_history: List[str] = Field(description="Smoking, alcohol, occupational hazards, relevant family conditions.")
    review_of_systems: List[str] = Field(description="Any other symptoms asked about and whether the patient affirmed or denied them.")

class Objective(BaseModel):
    vitals: List[str] = Field(description="Temperature, blood pressure, heart rate, weight, etc.")
    physical_exam_findings: List[str] = Field(description="Observations made or dictated by the physician (e.g., lungs clear, mild swelling).")
    diagnostic_results: List[str] = Field(description="Any rapid labs or imaging results discussed.")

class AssessmentPlan(BaseModel):
    diagnoses: List[str] = Field(description="Primary and secondary diagnoses or impressions.")
    orders_prescriptions: List[str] = Field(description="New medications prescribed, dosages, durations.")
    lab_imaging_orders: List[str] = Field(description="Tests the patient needs to get done.")
    patient_instructions: List[str] = Field(description="Advice given to the patient regarding care at home, diet, or symptom monitoring.")
    follow_up: Optional[str] = Field(description="When the patient should return.")

class ClinicalSOAPExtraction(BaseModel):
    administrative_data: AdministrativeData
    subjective: Subjective
    objective: Objective
    assessment_plan: AssessmentPlan


In [ ]:
# Initialize Gemini Model
llm = ChatVertexAI(
    model="gemini-3.1-flash-lite",
    temperature=0,
    location="global"
)

# Agent 1: Structured Extraction Chain
structured_llm = llm.with_structured_output(ClinicalSOAPExtraction)

extraction_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert medical AI assistant. Your task is to analyze clinical conversation transcripts and extract a highly structured JSON schema based on the SOAP framework."),
    ("human", "Please analyze the following conversation transcript and extract the structured clinical concepts:\n\n{transcript}")
])

agent_1_chain = extraction_prompt | structured_llm


In [ ]:
# Read the dummy transcript
with open('dummy_transcript_01.txt', 'r') as f:
    sample_transcript = f.read()

print("Original Transcript Loaded. Length:", len(sample_transcript))


In [ ]:
# Execute Agent 1
print("Agent 1: Extracting structured SOAP data...")
extracted_data = agent_1_chain.invoke({"transcript": sample_transcript})
print("\nExtraction complete.")
extracted_json_str = extracted_data.model_dump_json(indent=2)


In [ ]:
# --- Agent 2: Synthesize Case Sheet Summary ---

class PrescriptionItem(BaseModel):
    medicine: str = Field(description="Name of the medicine")
    dosage: str = Field(description="Dosage instructions (e.g., 0-0-1 After Food)")
    duration: str = Field(description="Duration of the medicine (e.g., 3 Days (Tot: 3 TAB))")

class CaseSheetSummary(BaseModel):
    patient_name: str = Field(description="Patient Name")
    gender: str = Field(description="Gender (e.g. Male, Female, Unknown)")
    age: str = Field(description="Age in years, months, days if possible, or just Years")
    patient_no: str = Field(description="A random or generated Patient No.")
    doctor: str = Field(description="Doctor's name")
    date: str = Field(description="Date and time of visit")
    chief_complaints: str = Field(description="Chief complaints summarized")
    vitals: str = Field(description="Vitals formatted nicely (e.g. Pulse: 99 bpm, BP: 135/95 mmHg...)")
    examination_findings: str = Field(description="Summary of examination findings")
    investigations: str = Field(description="Ordered tests or labs (e.g. BMP, MRI...)")
    diagnosis: str = Field(description="Diagnosis")
    prescriptions: List[PrescriptionItem] = Field(description="List of prescribed medicines")
    treatment_plan: str = Field(description="Summary of treatment plan")
    therapy_description: str = Field(description="Therapy required, or None")
    therapy_result: str = Field(description="Result of therapy, or N/A")
    notes: str = Field(description="Additional clinical notes")
    instructions: str = Field(description="Instructions to the patient (e.g. Avoid sweets, Return in 2 weeks)")

final_structured_llm = llm.with_structured_output(CaseSheetSummary)

final_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert clinical documentation AI. Your task is to generate a Patient Summary Report in a highly specific JSON format called 'CaseSheetSummary'.\n"
               "Rely on the provided JSON schema (SOAP data) for all clinical facts and structured data. "
               "Use the attached raw transcript ONLY to extract direct quotes, verify tone, or clarify ambiguities.\n"
               "Fill in all fields. If a field is missing, infer a sensible default or write 'N/A'."),
    ("human", "Here is the Structured Data (JSON):\n{json_data}\n\nHere is the Raw Transcript:\n{transcript}")
])

agent_2_chain = final_prompt | final_structured_llm


In [ ]:
# Execute Agent 2
print("Agent 2: Generating Case Sheet Summary data...")
case_sheet_data = agent_2_chain.invoke({
    "json_data": extracted_json_str,
    "transcript": sample_transcript
})

print("\nData synthesized successfully. Example field -> Chief Complaints:", case_sheet_data.chief_complaints)


In [ ]:
# --- Generate PDF Document ---

class CaseSheetPDF(FPDF):
    def header(self):
        # We simulate the Hospital header
        self.set_font('helvetica', 'B', 16)
        self.cell(0, 8, 'EZOVION MULTI SPECIALTY HOSPITAL', align='C', new_x="LMARGIN", new_y="NEXT")
        self.set_font('helvetica', '', 10)
        self.cell(0, 6, 'Chennai - 600044', align='C', new_x="LMARGIN", new_y="NEXT")
        self.ln(6)
        
        # Title
        self.set_font('helvetica', 'BU', 14)
        self.cell(0, 10, 'Case Sheet Summary', align='C', new_x="LMARGIN", new_y="NEXT")
        self.ln(2)
        
        # Simulate Barcode
        self.set_font('courier', '', 14)
        self.cell(0, 10, '||| |||| || |||||||| ||||', align='C', new_x="LMARGIN", new_y="NEXT")
        self.ln(4)

    def draw_section_line(self):
        self.set_draw_color(180, 180, 180)
        self.line(self.l_margin, self.get_y(), self.w - self.r_margin, self.get_y())
        self.ln(2)

def generate_pdf(data: CaseSheetSummary, output_filename="case_sheet_summary.pdf"):
    pdf = CaseSheetPDF()
    pdf.add_page()

    # Patient Details Grid
    y_before = pdf.get_y()
    
    # Left Column
    pdf.set_font('helvetica', '', 10)
    pdf.cell(25, 6, 'Name', new_x="RIGHT")
    pdf.set_font('helvetica', 'B', 10)
    pdf.cell(75, 6, data.patient_name, new_x="LMARGIN", new_y="NEXT")

    pdf.set_font('helvetica', '', 10)
    pdf.cell(25, 6, 'Gender', new_x="RIGHT")
    pdf.set_font('helvetica', 'B', 10)
    pdf.cell(75, 6, data.gender, new_x="LMARGIN", new_y="NEXT")

    pdf.set_font('helvetica', '', 10)
    pdf.cell(25, 6, 'Age', new_x="RIGHT")
    pdf.set_font('helvetica', 'B', 10)
    pdf.cell(75, 6, data.age, new_x="RIGHT")

    # Right Column
    pdf.set_xy(120, y_before)
    pdf.set_font('helvetica', '', 10)
    pdf.cell(25, 6, 'Patient No.', new_x="RIGHT")
    pdf.set_font('helvetica', 'B', 10)
    pdf.cell(0, 6, data.patient_no, new_x="LMARGIN", new_y="NEXT")

    pdf.set_xy(120, y_before + 6)
    pdf.set_font('helvetica', '', 10)
    pdf.cell(25, 6, 'Doctor', new_x="RIGHT")
    pdf.set_font('helvetica', 'B', 10)
    pdf.cell(0, 6, data.doctor, new_x="LMARGIN", new_y="NEXT")

    pdf.set_xy(120, y_before + 12)
    pdf.set_font('helvetica', '', 10)
    pdf.cell(25, 6, 'Date', new_x="RIGHT")
    pdf.set_font('helvetica', 'B', 10)
    pdf.cell(0, 6, data.date, new_x="LMARGIN", new_y="NEXT")

    pdf.ln(4)
    pdf.draw_section_line()

    # Helper for label: value
    def add_field(label, value):
        pdf.set_font('helvetica', 'B', 10)
        pdf.cell(pdf.get_string_width(label) + 2, 8, label, new_x="RIGHT")
        pdf.set_font('helvetica', '', 10)
        pdf.multi_cell(0, 8, value, new_x="LMARGIN", new_y="NEXT")
        pdf.draw_section_line()

    add_field('Chief Complaints :', data.chief_complaints)
    add_field('Vitals :', data.vitals)
    add_field('Examination : Findings :', data.examination_findings)
    add_field('Investigation :', data.investigations)
    add_field('Diagnosis :', data.diagnosis)

    # Prescription Section
    pdf.set_font('helvetica', 'B', 10)
    pdf.cell(0, 8, 'Prescription :', new_x="LMARGIN", new_y="NEXT")
    pdf.ln(1)
    
    # Table Header
    pdf.set_font('helvetica', 'B', 10)
    pdf.cell(10, 8, '#', new_x="RIGHT")
    pdf.cell(80, 8, 'Medicine', new_x="RIGHT")
    pdf.cell(50, 8, 'Dosage', new_x="RIGHT")
    pdf.cell(0, 8, 'Duration', new_x="LMARGIN", new_y="NEXT")

    # Table Rows
    pdf.set_font('helvetica', '', 10)
    for idx, p in enumerate(data.prescriptions, 1):
        pdf.cell(10, 8, str(idx), new_x="RIGHT")
        pdf.cell(80, 8, p.medicine, new_x="RIGHT")
        pdf.cell(50, 8, p.dosage, new_x="RIGHT")
        pdf.cell(0, 8, p.duration, new_x="LMARGIN", new_y="NEXT")
    
    pdf.ln(2)
    pdf.draw_section_line()

    add_field('Treatment Plan :', data.treatment_plan)

    # Therapy Section
    pdf.set_font('helvetica', 'B', 10)
    pdf.cell(0, 8, 'Therapy', new_x="LMARGIN", new_y="NEXT")
    
    pdf.cell(100, 8, 'Description', new_x="RIGHT")
    pdf.cell(0, 8, 'Result', new_x="LMARGIN", new_y="NEXT")
    
    pdf.set_font('helvetica', '', 10)
    pdf.cell(100, 8, data.therapy_description, new_x="RIGHT")
    pdf.cell(0, 8, data.therapy_result, new_x="LMARGIN", new_y="NEXT")

    pdf.ln(2)
    pdf.set_font('helvetica', 'B', 10)
    pdf.cell(0, 8, 'Notes', new_x="LMARGIN", new_y="NEXT")
    pdf.set_font('helvetica', '', 10)
    pdf.multi_cell(0, 8, data.notes, new_x="LMARGIN", new_y="NEXT")
    pdf.draw_section_line()

    add_field('Instructions :', data.instructions)

    pdf.output(output_filename)
    print(f"PDF generated successfully: {output_filename}")

# Execute PDF generation
generate_pdf(case_sheet_data)
